In [ ]:
%cd ..

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [2]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [3]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [ ]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/finance-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "finance_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/finance_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [5]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [6]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [76]:
SHARED_SOC_REVENUE_MAPPING_FINAL = {
    # ===== Time =====
    "Ngày xuất hóa đơn": "invoice_date",
    "Tháng": "invoice_month",

    # ===== Invoice =====
    "Xuất hóa đơn (HD)/ tạm tính (TT)": "invoice_type",
    "Nội dung đề nghị TT(Theo chứng từ gốc)": "billing_description",

    # ===== Revenue =====
    "Doanh thu": "revenue_amount",
    "Chia sẻ": "revenue_share_amount",
    "Doanh thu cuối": "final_revenue_amount",

    "VAT": "vat_amount",
    "Tiền hàng (đã bao gồm VAT)": "gross_amount",

    # ===== Revenue behavior =====
    "DT dòng tiền đều/ DT lên 1 lần": "revenue_recognition_type",

    # ===== Product =====
    "Phân loại SP/DV": "service_category",
    "SPDV cụ thể": "service_name",
    "Mã SPDV": "service_code",

    # ===== Customer =====
    "Khách hàng": "customer_name",
    "Phân loại KH": "customer_type",
    "Kênh khách hàng": "customer_channel",
    "Segment": "customer_segment",
    "Nhóm khách hàng": "customer_group",

    # ===== Market =====
    "Nội bộ/ Ngoài/QTế/Thị Trường": "customer_scope",

    # ===== Org =====
    "Phòng": "department",
    "AM": "account_manager",
    "AM hiện tại": "current_account_manager",
    "Presale": "presale",

    # ===== Classification =====
    "Phân loại DT Cũ/Mới": "revenue_type",
    "Phân loại SOC (SOC và non-SOC)": "soc_type",

    # ===== Revenue sharing =====
    "DT chia sẻ từ MSS": "mss_shared_revenue",

    # ===== Region =====
    "Nam/ Bắc": "region",
    "Bắc/Nam/Thị trường": "market_scope",

    # ===== Flags =====
    "Vvip": "is_vvip_customer",
    "HĐ khung": "is_master_contract",

    # ===== Note =====
    "Ghi chú": "note",
    "Note": "note",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 1_2026\Doanh_thu_cong_ty_t1_2026_SOC_chia_se_cho_SPDV.xlsx"

df1 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=SHARED_SOC_REVENUE_MAPPING_FINAL,
    header_row=0,
    drop_rows=1,
)
df1["source_file"] = filename.split("\\")[-1]

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Dữ liệu dashboard kinh doanh tháng 2_2026\Doanh_thu_cong_ty_t2_2026_SOC_chia_se_cho_SPDV.xlsx"

df2 = read_excel_and_normalize_columns(
      filename,
    sheet_name="Sheet1",
    mapping=SHARED_SOC_REVENUE_MAPPING_FINAL,
    header_row=0,
    drop_rows=1,
)
df2["source_file"] = filename.split("\\")[-1]
df = pd.concat([df1, df2], axis=0)

# resource_name = "costs"
resource_name = "sales_revenue_share_soc_product_category"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


704
✅ Done: sales_revenue_share_soc_product_category.json created


In [77]:
resource_name = "sales_revenue_share_soc_product_category"
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  sales_revenue_share_soc_product_category
sales_revenue_share_soc_product_category
Replace Upload  s3a://vcs-raw/finance-raw/sales_revenue_share_soc_product_category ./tmp/data/sales_revenue_share_soc_product_category/data_sales_revenue_share_soc_product_category_20260318_211730.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/finance-raw/sales_revenue_share_soc_product_category
Uploaded SQL definition


False